# Tokenization: Byte-Pair Encoding by Hand

Every transformer walkthrough in this section — [[bert-on-eeg]],
[[autoregressive-transformer-encoder-decoder]], [[kimi-k3]],
[[deepseek-v4]] — begins with a sequence of token IDs already in hand. This page
is the step before that: **how text becomes those integers**.

It is not a detail. Tokenization decides how long your sequences are (and
attention cost is quadratic or worse in that length), which words the model can
never spell, why it struggles to count letters, and why the same sentence costs
three times as much in one language as another.

The dominant method is **Byte-Pair Encoding**. It was a data compression
algorithm from 1994, repurposed for NLP in 2015, and it is simple enough to run
completely by hand — which is what this page does, merge by merge.

## 1. The problem BPE solves

Two obvious approaches, both bad:

**One token per word.** The vocabulary is unbounded — every name, typo, compound
and inflection is a new entry. Anything unseen becomes `<UNK>` and the model is
blind to it.

**One token per character.** The vocabulary is tiny and nothing is ever unknown,
but sequences become very long, and the model must rediscover that `t-h-e` is a
unit every single time.

BPE interpolates. Start from characters, then **repeatedly merge the most
frequent adjacent pair** into a new symbol. Common words end up as single tokens;
rare words survive as a handful of pieces; nothing is ever out-of-vocabulary. You
choose where on that spectrum to sit by choosing how many merges to perform.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=140)


def banner(title, ch="="):
    print(ch * 78)
    print(f"  {title}")
    print(ch * 78)


def step(label):
    print()
    print(label)
    print("=" * 78)


def check(manual, from_array, label="MATCH"):
    ok = np.allclose(manual, from_array)
    print(f"      = {manual:+.6f}   |  from array: {from_array:+.6f}   |  {label}: {ok}")
    return ok


def expand(u, v, per_line=4):
    """Write a dot product out term by term."""
    terms = [f"({a:+.4f})*({b:+.4f})" for a, b in zip(u, v)]
    lines, out = [], ""
    for i, t in enumerate(terms):
        out += t + (" + " if i < len(terms) - 1 else "")
        if (i + 1) % per_line == 0:
            lines.append(out); out = ""
    if out:
        lines.append(out)
    return "\n      ".join(lines)


def show_matrix(M, name="M", rows=None, cols=None, fmt="{:+8.3f}", max_cols=14):
    M = np.asarray(M)
    if M.ndim == 1:
        M = M[None, :]
    cut = M.shape[1] > max_cols
    Ms = M[:, :max_cols] if cut else M
    print(f"{name} {np.asarray(M).shape}" +
          (f"   [first {max_cols} of {M.shape[1]} columns]" if cut else "") + " =")
    if cols is not None:
        head = "".join(f"{str(c):>10}" for c in (cols[:max_cols] if cut else cols))
        print(" " * 13 + head)
    for r in range(Ms.shape[0]):
        lab = f"{str(rows[r]):>10} | " if rows is not None else f"{r:>10} | "
        print("   " + lab + "  ".join(fmt.format(v) for v in Ms[r]))


def show_matmul(A, B, C, na="A", nb="B", nc="C", i=0, j=0,
                rows_A=None, note=None, max_cols=14):
    """Print both operands of A @ B = C, then verify one element by hand."""
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{na} {A.shape}   @   {nb} {B.shape}   =   {nc} {C.shape}")
    print(f"  shape rule: ({A.shape[0]} x {A.shape[1]}) @ ({B.shape[0]} x {B.shape[1]})"
          f" -> ({A.shape[0]} x {B.shape[1]})   inner dim {A.shape[1]} cancels")
    if note:
        print(f"  {note}")
    print("=" * 78)
    print()
    show_matrix(A, na, rows_A, max_cols=max_cols)
    print()
    show_matrix(B, nb, max_cols=max_cols)
    print()
    show_matrix(C, nc, rows_A, max_cols=max_cols)
    print(f"\n  worked element  {nc}[{i},{j}]  =  row {i} of {na}  .  column {j} of {nb}")
    print(f"      row {i} of {na} = {np.round(A[i], 4)}")
    print(f"      col {j} of {nb} = {np.round(B[:, j], 4)}")
    print(f"      {expand(A[i], B[:, j])}")
    check(float(A[i] @ B[:, j]), float(C[i, j]))


def show_elementwise(A, B, C, na, nb, nc, op="+", rows=None):
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{nc} = {na} {op} {nb}      all three are {A.shape} - element-wise")
    for r in range(A.shape[0]):
        lab = f"{rows[r]:>10}" if rows is not None else f"{r:>10}"
        print(f"\n{lab} |")
        print("   " + f"{na:>14}: " + "  ".join(f"{v:+8.3f}" for v in A[r]))
        print("   " + f"{nb:>14}: " + "  ".join(f"{v:+8.3f}" for v in B[r]))
        print("   " + f"{nc:>14}: " + "  ".join(f"{v:+8.3f}" for v in C[r]))
    print(f"\n  check [0,0]: {A[0,0]:+.6f} {op} {B[0,0]:+.6f} = {C[0,0]:+.6f}   "
          f"MATCH: {np.isclose(A[0,0] + B[0,0] if op == '+' else A[0,0] - B[0,0], C[0,0])}")


print("Display helpers ready: banner, step, check, expand, show_matrix, "
      "show_matmul, show_elementwise.")

from collections import Counter

banner("THE TWO EXTREMES, ON A SMALL CORPUS")

corpus = ("the cat sat on the mat the cat ate the rat "
          "a cat and a rat sat on a mat the rats ate")
words = corpus.split()
print(f"  corpus: {len(words)} words, {len(set(words))} distinct")
print(f"  \"{corpus}\"")

step("Word-level")
vocab_w = sorted(set(words))
print(f"  vocabulary size : {len(vocab_w)}")
print(f"  sequence length : {len(words)} tokens")
print(f"  vocab: {vocab_w}")
print(f"  'rats' seen? {'rats' in vocab_w}. But 'cats' would be <UNK>:"
      f" {'cats' not in vocab_w}")

step("Character-level")
vocab_c = sorted(set(corpus))
print(f"  vocabulary size : {len(vocab_c)}")
print(f"  sequence length : {len(corpus)} tokens"
      f"  ({len(corpus)/len(words):.1f}x longer)")
print(f"  nothing is ever unknown, but 'the' is 3 tokens every time it appears")
print(f"  and it appears {words.count('the')} times = {words.count('the')*3} tokens"
      f" spent on one word.")

step("The cost of sequence length in a transformer")
print(f"  attention is O(n^2) in the sequence length, so:")
print(f"  {'representation':>16}  {'n':>5}  {'n^2':>8}")
print("  " + "-" * 34)
for lab, nn in [('word-level', len(words)), ('char-level', len(corpus))]:
    print(f"  {lab:>16}  {nn:>5}  {nn*nn:>8,}")
print("  " + "-" * 34)
print(f"  {len(corpus)**2/len(words)**2:.1f}x more attention work for the same text.")
print(f"  This is why tokenization is an efficiency decision, not just a")
print(f"  preprocessing one.")

Display helpers ready: banner, step, check, expand, show_matrix, show_matmul, show_elementwise.
  THE TWO EXTREMES, ON A SMALL CORPUS
  corpus: 23 words, 10 distinct
  "the cat sat on the mat the cat ate the rat a cat and a rat sat on a mat the rats ate"

Word-level
  vocabulary size : 10
  sequence length : 23 tokens
  vocab: ['a', 'and', 'ate', 'cat', 'mat', 'on', 'rat', 'rats', 'sat', 'the']
  'rats' seen? True. But 'cats' would be <UNK>: True

Character-level
  vocabulary size : 12
  sequence length : 84 tokens  (3.7x longer)
  nothing is ever unknown, but 'the' is 3 tokens every time it appears
  and it appears 5 times = 15 tokens spent on one word.

The cost of sequence length in a transformer
  attention is O(n^2) in the sequence length, so:
    representation      n       n^2
  ----------------------------------
        word-level     23       529
        char-level     84     7,056
  ----------------------------------
  13.3x more attention work for the same text.
  This is wh

## 2. Training BPE, merge by merge

The training algorithm:

1. Split every word into characters. Append an end-of-word marker `_` so the
   algorithm can tell `est` at the end of a word from `est` in the middle.
2. Count every adjacent symbol pair across the corpus, weighted by word frequency.
3. Merge the most frequent pair into a single new symbol.
4. Record the merge and repeat.

The **merge list, in order**, *is* the trained tokenizer. There is nothing else
to it — no neural network, no probabilities. Applying it to new text means
replaying those merges in the same order.

In [2]:
banner("TRAINING BPE, ONE MERGE AT A TIME")

def get_stats(vocab):
    """Count adjacent pairs, weighted by word frequency."""
    pairs = Counter()
    for word, freq in vocab.items():
        syms = word.split()
        for i in range(len(syms)-1):
            pairs[(syms[i], syms[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    out, bigram = {}, ' '.join(pair)
    joined = ''.join(pair)
    for word, freq in vocab.items():
        out[word.replace(bigram, joined)] = freq
    return out

# word -> frequency, each word spelled out as space-separated symbols
wf = Counter(words)
vocab = {' '.join(list(w) + ['_']): f for w, f in wf.items()}

step("Starting point: every word spelled out")
for w, f in sorted(vocab.items(), key=lambda kv: -kv[1]):
    print(f"    {f:>2} x   {w}")

merges = []
N_MERGES = 8
for step_i in range(1, N_MERGES+1):
    stats = get_stats(vocab)
    if not stats:
        break
    best, cnt = stats.most_common(1)[0]
    top3 = stats.most_common(3)
    step(f"Merge {step_i}")
    print(f"  most frequent adjacent pairs:")
    for (a, b), c in top3:
        mark = "  <- merging this" if (a, b) == best else ""
        print(f"    ({a!r:>6}, {b!r:>6}) : {c}{mark}")
    merges.append(best)
    vocab = merge_vocab(best, vocab)
    print(f"  new symbol: {''.join(best)!r}")
    changed = [w for w in vocab if ''.join(best) in w]
    print(f"  words containing it now: "
          f"{[w for w in changed][:5]}")

step("The trained tokenizer IS this ordered list")
for i, (a, b) in enumerate(merges, 1):
    print(f"  {i:>2}. {a!r:>8} + {b!r:>8}  ->  {''.join((a,b))!r}")

step("Final segmentation of each word")
for w, f in sorted(vocab.items(), key=lambda kv: -kv[1]):
    print(f"    {f:>2} x   {w.split()}")

  TRAINING BPE, ONE MERGE AT A TIME

Starting point: every word spelled out
     5 x   t h e _
     3 x   c a t _
     3 x   a _
     2 x   s a t _
     2 x   o n _
     2 x   m a t _
     2 x   a t e _
     2 x   r a t _
     1 x   a n d _
     1 x   r a t s _

Merge 1
  most frequent adjacent pairs:
    (   'a',    't') : 12  <- merging this
    (   't',    '_') : 9
    (   'e',    '_') : 7
  new symbol: 'at'
  words containing it now: ['c at _', 's at _', 'm at _', 'at e _', 'r at _']

Merge 2
  most frequent adjacent pairs:
    (  'at',    '_') : 9  <- merging this
    (   'e',    '_') : 7
    (   't',    'h') : 5
  new symbol: 'at_'
  words containing it now: ['c at_', 's at_', 'm at_', 'r at_']

Merge 3
  most frequent adjacent pairs:
    (   'e',    '_') : 7  <- merging this
    (   't',    'h') : 5
    (   'h',    'e') : 5
  new symbol: 'e_'
  words containing it now: ['t h e_', 'at e_']

Merge 4
  most frequent adjacent pairs:
    (   't',    'h') : 5  <- merging this
    (   

## 3. Applying the tokenizer to new text

Encoding replays the merges **in training order**. Order matters: a later merge
may depend on a symbol that an earlier one created, so applying them out of order
gives a different — and wrong — segmentation.

The important behaviour is what happens to a word the tokenizer has never seen.
It does not fail. It falls back to whatever pieces it does know, which is why BPE
has no `<UNK>` token at all.

In [3]:
banner("ENCODING NEW TEXT")

def encode(word, merges):
    syms = list(word) + ['_']
    trace = [list(syms)]
    for a, b in merges:                      # training order matters
        i = 0
        while i < len(syms)-1:
            if syms[i] == a and syms[i+1] == b:
                syms[i:i+2] = [a+b]
            else:
                i += 1
        if syms != trace[-1]:
            trace.append(list(syms))
    return syms, trace

step("A word from the training corpus")
out, trace = encode('the', merges)
for i, t in enumerate(trace):
    lab = "start" if i == 0 else f"after a merge"
    print(f"    {lab:>14}: {t}")
print(f"  'the' -> {out}   ({len(out)} token{'s' if len(out)!=1 else ''})")

step("A word the tokenizer has NEVER seen")
for w in ['cats', 'chat', 'strategy']:
    out, _ = encode(w, merges)
    print(f"  {w!r:>12} -> {out}   ({len(out)} tokens)")
print()
print(f"  No <UNK> anywhere. Unseen words decompose into known pieces, and in")
print(f"  the worst case into single characters -- which are always in the")
print(f"  vocabulary because that is where training started.")

step("Why merge ORDER matters")
out_fwd, _ = encode('the', merges)
out_rev, _ = encode('the', list(reversed(merges)))
print(f"  merges applied in training order : {out_fwd}")
print(f"  merges applied in reverse order  : {out_rev}")
print(f"  same result? {out_fwd == out_rev}")
print(f"  The list is a sequence of dependent rewrites, not a set of rules.")

  ENCODING NEW TEXT

A word from the training corpus
             start: ['t', 'h', 'e', '_']
     after a merge: ['t', 'h', 'e_']
     after a merge: ['th', 'e_']
     after a merge: ['the_']
  'the' -> ['the_']   (1 token)

A word the tokenizer has NEVER seen
        'cats' -> ['c', 'at', 's', '_']   (4 tokens)
        'chat' -> ['c', 'h', 'at_']   (3 tokens)
    'strategy' -> ['s', 't', 'r', 'at', 'e', 'g', 'y', '_']   (8 tokens)

  No <UNK> anywhere. Unseen words decompose into known pieces, and in
  the worst case into single characters -- which are always in the
  vocabulary because that is where training started.

Why merge ORDER matters
  merges applied in training order : ['the_']
  merges applied in reverse order  : ['th', 'e_']
  same result? False
  The list is a sequence of dependent rewrites, not a set of rules.


## 4. The consequences that show up in real models

Everything odd about how language models handle text traces back to this
algorithm.

**Counting letters is hard.** If `strawberry` is three tokens, the model never
sees individual letters, so "how many r's" is a question about something it
cannot observe directly.

**Arithmetic is fragile.** Whether `1234` is one token, or `12`+`34`, or
`1`+`2`+`3`+`4`, depends on which digit strings happened to be frequent in the
training corpus. Digit grouping is arbitrary, which is why many recent
tokenizers special-case digits.

**Some languages cost far more.** BPE merges are learned from a corpus. A
language under-represented in that corpus gets few merges, so its text stays
close to character-level — more tokens for the same meaning, more compute, and
less context.

**Whitespace attaches to words.** Most modern tokenizers make `" the"` (with the
leading space) a different token from `"the"`, which is why prompts ending in a
trailing space often behave strangely.

In [4]:
banner("THE CONSEQUENCES, MEASURED")

step("Fertility: tokens per word, for text the tokenizer knows well or badly")
seen   = "the cat sat on the mat"
unseen = "quixotic zephyr bureaucracy"
for lab, txt in [('in-corpus', seen), ('out-of-corpus', unseen)]:
    toks = [t for w in txt.split() for t in encode(w, merges)[0]]
    print(f"  {lab:>14}: {len(txt.split())} words -> {len(toks)} tokens"
          f"   (fertility {len(toks)/len(txt.split()):.2f})")
print()
print(f"  Same tokenizer, {len(txt.split())} words each. The corpus the merges")
print(f"  were learned from decides the cost of the text you actually send.")

step("How vocabulary size trades against sequence length")
print(f"  Re-training with different merge budgets on the same corpus:")
print(f"  {'merges':>7}  {'vocab size':>11}  {'tokens for corpus':>18}"
      f"  {'tokens/word':>12}")
print("  " + "-" * 54)
base_symbols = set(''.join(wf)) | {'_'}       # every character, plus the marker
for nm in [0, 4, 8, 16, 32, 64]:
    v = {' '.join(list(w) + ['_']): f for w, f in wf.items()}
    ms = []
    for _ in range(nm):
        s = get_stats(v)
        if not s:
            break
        b = s.most_common(1)[0][0]
        ms.append(b)
        v = merge_vocab(b, v)
    total = sum(len(encode(w, ms)[0]) for w in words)
    # The vocabulary is the base alphabet plus one new symbol per merge
    # PERFORMED. Counting symbols left in the final words instead would
    # undercount: a symbol that got merged into a larger one is still in the
    # vocabulary, it just no longer appears on its own.
    vsize = len(base_symbols) + len(ms)
    note = "  (corpus exhausted)" if len(ms) < nm else ""
    print(f"  {nm:>7}  {vsize:>11}  {total:>18}  {total/len(words):>12.2f}{note}")
print("  " + "-" * 54)
print(f"  More merges -> bigger vocabulary -> shorter sequences. That is the")
print(f"  entire design trade: embedding table size against attention cost.")
print(f"  Real tokenizers sit around 32k-256k merges; GPT-2 used 50,257")
print(f"  tokens, Llama 3 used 128k, and the trend is upward as context")
print(f"  windows grow and sequence length gets more expensive.")

step("Counting letters, from the model's point of view")
for w in ['rat', 'rats', 'strategy']:
    toks, _ = encode(w, merges)
    print(f"  {w!r:>10} -> {toks}")
    print(f"  {'':>10}    the model sees {len(toks)} opaque symbol"
          f"{'s' if len(toks)!=1 else ''}, not {len(w)} letters")
print()
print(f"  Asking 'how many t's in strategy' requires information that was")
print(f"  discarded before the first embedding lookup.")

  THE CONSEQUENCES, MEASURED

Fertility: tokens per word, for text the tokenizer knows well or badly
       in-corpus: 6 words -> 9 tokens   (fertility 1.50)
   out-of-corpus: 3 words -> 28 tokens   (fertility 9.33)

  Same tokenizer, 3 words each. The corpus the merges
  were learned from decides the cost of the text you actually send.

How vocabulary size trades against sequence length
  Re-training with different merge budgets on the same corpus:
   merges   vocab size   tokens for corpus   tokens/word
  ------------------------------------------------------
        0           12                  85          3.70
        4           16                  52          2.26
        8           20                  39          1.70
       16           28                  26          1.13
       32           31                  23          1.00  (corpus exhausted)
       64           31                  23          1.00  (corpus exhausted)
  ------------------------------------------------

## 5. What to take away

**BPE is a compression algorithm, and the merge list is the whole model.** No
probabilities, no training objective — just "merge the most frequent adjacent
pair, repeatedly". The ordered list of merges is the tokenizer.

**It has no unknown words**, because it can always fall back to characters. That
single property is why it replaced word-level vocabularies.

**Vocabulary size and sequence length are the same dial.** More merges means
longer tokens, shorter sequences, and a larger embedding table. Since attention
costs $O(n^2)$ — see [[deepseek-v4]] for how far people go to beat that — the
pressure has been steadily toward larger vocabularies.

**Most "weird" LLM behaviour around letters, digits, and non-English text is a
tokenizer artefact**, not a reasoning failure. The information was destroyed
before the model saw anything.

### Verified in this notebook

- Eight merges performed by hand, each showing the pair counts that motivated it
- The resulting merge list applied to training words and to words never seen, with no `<UNK>` produced
- Applying the same merges in reverse order gives a different segmentation, showing the list is order-dependent
- Fertility measured on in-corpus vs out-of-corpus text
- Merge budget from 0 to 64 traded against vocabulary size and total tokens

### Where this connects

This is the step immediately before every transformer page:
[[autoregressive-transformer-encoder-decoder]] takes these IDs and embeds them,
[[bert-on-eeg]] replaces them with signal patches, and [[kimi-k3]] and
[[deepseek-v4]] are both largely concerned with making long token sequences
affordable.